# CiteCheck: fine-tune the verifier on a Colab GPU

**Runtime → Change runtime type → T4 GPU** before running anything.

Every number comes out of `eval.run_eval`, the same harness as Session 5, so runs are directly comparable with
`zero_shot_nli` (dev macro-F1 0.603, false-SUPPORT 0.108). Rules that still apply here:

* training reads `claims_train.jsonl` only; the 200 claims tau is tuned on are **held out** (run_eval refuses otherwise);
* dev is for reporting only: never pick hyperparameters by looking at dev numbers you then report as final;
* commit `eval/results/<run>/`, never `models/` (weights go to Drive or the Hugging Face Hub).

In [ ]:
# 1. settings: point these at your fork and branch
FORK_URL = "https://github.com/Lord2709/citecheck"   # <- your fork
BRANCH   = "feat/modeling-finetune-v2"
SAVE_TO_DRIVE = True                                 # keep checkpoints across Colab sessions
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. code + dependencies + data  (~3 min)
import os
if not os.path.exists("citecheck"):
    !git clone -b {BRANCH} {FORK_URL} citecheck
%cd citecheck
!git pull -q
!pip install -q -r requirements.txt
!python -m data.fetch_scifact

In [ ]:
# 3. where checkpoints live
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS = "/content/drive/MyDrive/citecheck-models"
else:
    MODELS = "models"
os.makedirs(MODELS, exist_ok=True)
print(MODELS)

## Experiment A: fine-tune the deployed base model (~10-15 min on a T4)

Same model the app runs today (`DeBERTa-v3-base-mnli-fever-anli`), now trained on SciFact train. Watch the
*held-out pair macro-F1* printed after each epoch: the best epoch is the one that is saved.

In [ ]:
if not os.path.exists("eval/results/finetuned_nli/results.json"):   # already run and committed? skip it
  !python -m eval.finetune_nli --out {MODELS}/ft-base --epochs 3 --batch-size 16 --lr 2e-5 && \
    python -m eval.run_eval --name finetuned_nli --retriever bm25 --verifier nli --model {MODELS}/ft-base \
      --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/lexical

## Experiment B: bigger zero-shot model, no training (~5 min)

Without this row we can't tell whether a gain in C comes from training or just from model size.
Before reporting it, check the model's license on its Hugging Face card and add it to `data/README.md`.

In [ ]:
LARGE = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
if not os.path.exists("eval/results/zero_shot_nli_large/results.json"):   # already run? skip it
  !python -m eval.run_eval --name zero_shot_nli_large --retriever bm25 --verifier nli --model {LARGE} \
    --batch-size 8 --oracle --compare-with eval/results/zero_shot_nli

## Experiment C: fine-tune the large model (~35-50 min on a T4)

Batch 4 x 4 accumulation = effective batch 16; gradient checkpointing keeps it inside 15 GB.
If Colab disconnects, the checkpoint of the best finished epoch is already on Drive.

In [ ]:
if not os.path.exists("eval/results/finetuned_nli_large/results.json"):
  !python -m eval.finetune_nli --base {LARGE} --out {MODELS}/ft-large --epochs 2 --batch-size 4 --grad-accum 4 \
      --lr 6e-6 --gradient-checkpointing && \
    python -m eval.run_eval --name finetuned_nli_large --retriever bm25 --verifier nli --model {MODELS}/ft-large \
      --batch-size 8 --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/zero_shot_nli_large

## Experiment D: judge the whole abstract (no sentence selection)

Same base model and settings as A (effective batch 16, 3 epochs, lr 2e-5); only the evidence unit changes.
The zero-shot row shows how much comes from the unit alone. ~15 min.

In [ ]:
if not os.path.exists("eval/results/finetuned_nli_abstract/results.json"):   # already run and committed? skip it
  !python -m eval.run_eval --name zero_shot_nli_abstract --retriever bm25 --verifier nli --evidence-unit abstract \
      --oracle --compare-with eval/results/zero_shot_nli
  !python -m eval.finetune_nli --unit abstract --out {MODELS}/ft-base-abstract --epochs 3 --batch-size 8 --grad-accum 2 --lr 2e-5 && \
    python -m eval.run_eval --name finetuned_nli_abstract --retriever bm25 --verifier nli --evidence-unit abstract \
      --model {MODELS}/ft-base-abstract --oracle --compare-with eval/results/finetuned_nli --compare-with eval/results/zero_shot_nli


## Experiment E: judge each sentence, take the most decisive one

One verifier call per sentence (~8x more calls than A, but short ones); the deciding sentences become the evidence
shown to the user. ~3x more training pairs than A, so ~10 min.

In [ ]:
if not os.path.exists("eval/results/finetuned_nli_sentence/results.json"):   # already run and committed? skip it
  !python -m eval.run_eval --name zero_shot_nli_sentence --retriever bm25 --verifier nli --evidence-unit sentence \
      --batch-size 64 --oracle --compare-with eval/results/zero_shot_nli
  !python -m eval.finetune_nli --unit sentence --out {MODELS}/ft-base-sentence --epochs 3 --batch-size 16 --lr 2e-5 && \
    python -m eval.run_eval --name finetuned_nli_sentence --retriever bm25 --verifier nli --evidence-unit sentence \
      --model {MODELS}/ft-base-sentence --batch-size 64 --oracle \
      --compare-with eval/results/finetuned_nli --compare-with eval/results/finetuned_nli_abstract --compare-with eval/results/zero_shot_nli


## Experiment F: hybrid stage 2 (a small local LLM reads, Python does the maths)

Stage 1 is the fine-tuned base model from A (`ft-base` on Drive). Claims it is not confident about (abstained, or
strongest paper < 0.98) go to a small local LLM that extracts the relevant sentences, quantities, p-values and CIs as
JSON; Python grounds every number in the text and runs the significance / quantity checks. A failed check can turn a
verdict into CONTRADICT; a check can never create a SUPPORT.

First cell re-evaluates A unchanged (same numbers) so it also saves its cited-paper predictions for the comparison.
The hybrid run generates once (~30-45 min on a T4); the `checks` and `llm` ablations reuse the cached generations.
Check the LLM's license on its model card before reporting (Qwen3: Apache-2.0; Phi-4-mini: MIT).

In [ ]:
!python -m eval.run_eval --name finetuned_nli --retriever bm25 --verifier nli --model {MODELS}/ft-base \
    --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/lexical

In [ ]:
import json
LLM = "Qwen/Qwen3-4B"          # alternatives: "microsoft/Phi-4-mini-instruct"; anything instruction-tuned, ~2-8B
S2 = f"--retriever bm25 --verifier nli --model {MODELS}/ft-base --oracle --reasoner {LLM} --reasoner-batch-size 8"
if not os.path.exists("eval/results/hybrid_nli_checks/results.json"):   # ~45 min; skip once committed
  !python -m eval.run_eval --name hybrid_nli {S2} --reasoner-mode hybrid --compare-with eval/results/finetuned_nli
  !python -m eval.run_eval --name hybrid_nli_checks {S2} --reasoner-mode checks --compare-with eval/results/finetuned_nli
  !python -m eval.run_eval --name hybrid_nli_llm {S2} --reasoner-mode llm --compare-with eval/results/finetuned_nli
for run in [r for r in ["finetuned_nli", "hybrid_nli", "hybrid_nli_checks", "hybrid_nli_llm"] if os.path.exists(f"eval/results/{r}/results.json")]:
    r = json.load(open(f"eval/results/{run}/results.json")); cl, cp = r["claim_level"], r["cited_paper"]
    print(f"{run:18s} e2e F1 {cl['macro_f1']:.3f} FS {cl['false_support_rate']:.3f} | cited-paper F1 {cp['macro_f1']:.3f} "
          f"{[round(x, 3) for x in cp['macro_f1_ci95']]} FS {cp['false_support_rate']:.3f}  "
          + "  ".join(f"vs {c['vs']} {c['diff']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}]" for c in cp.get("paired_comparisons", [])))


## Ship it: put the fine-tuned verifier on the Hugging Face Hub and promote it

The app downloads its model by name, so the checkpoint on Drive has to go to the Hub first.

1. Make a free account at huggingface.co. Under **Settings → Access Tokens**, create a token with **Write** access.
2. Run the next cell and paste the token when it asks. It uploads `ft-base`, including `train_log.json`; the
   evaluation harness reads that log from the Hub to check the tau-tuning claims were held out.
3. The cell after it re-scores the Hub copy on dev (same weights, so the same 0.693) and, with `--promote`, writes
   `eval/results/deployed_config.json`. That's the file the app reads.

In [ ]:
HF_USER = "your-hf-username"               # <- your Hugging Face username
HUB_MODEL = f"{HF_USER}/citecheck-nli-ft-base"
from huggingface_hub import login, create_repo, upload_folder
login()
card = f"""---
license: mit
base_model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
datasets: [allenai/scifact]
pipeline_tag: text-classification
---
# CiteCheck verifier (DeBERTa-v3-base fine-tuned on SciFact)

Judges whether evidence sentences from a paper's abstract SUPPORT, CONTRADICT or are NEUTRAL to a scientific claim.
Fine-tuned on SciFact train (CC BY 4.0) minus 200 claims held out for threshold tuning; see train_log.json.
SciFact dev, end to end in CiteCheck: macro-F1 0.693 (95% CI 0.636-0.749), false-SUPPORT rate 0.068.
Biomedical abstracts only; not validated on other fields. Course project (UMD DATA/MSML 641).
"""
open(f"{MODELS}/ft-base/README.md", "w").write(card)
create_repo(HUB_MODEL, exist_ok=True)
upload_folder(folder_path=f"{MODELS}/ft-base", repo_id=HUB_MODEL)
print("uploaded:", f"https://huggingface.co/{HUB_MODEL}")

In [ ]:
!python -m eval.run_eval --name finetuned_nli --retriever bm25 --verifier nli --model {HUB_MODEL} \
    --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/lexical --promote
!cat eval/results/deployed_config.json

## Experiment G: learned evidence-sentence selector (step 5)

A small cross-encoder is trained to pick the gold rationale sentences, with the tau-tuning claims held out as before.
Word overlap gets a complete rationale into its top 3 for 79% of dev gold papers, and into its top 1 for only 47%.
Same verifier as A; only the sentence selection changes. About 10 minutes.

In [ ]:
!python -m eval.train_selector --out {MODELS}/selector --epochs 3
!python -m eval.run_eval --name finetuned_nli_selector --retriever bm25 --verifier nli --model {MODELS}/ft-base \
    --sentence-scorer cross:{MODELS}/selector --oracle --compare-with eval/results/finetuned_nli
!python -m eval.run_eval --name finetuned_nli_selector_m2 --retriever bm25 --verifier nli --model {MODELS}/ft-base \
    --sentence-scorer cross:{MODELS}/selector --sentences 2 --oracle --compare-with eval/results/finetuned_nli

## Real citations (step 3): collect the citing sentences

Fetches real sentences from published papers that cite papers in our SciFact corpus (excluding any paper a train claim
uses). The file stays out of git, because it is Semantic Scholar text. Download it and send it over for labelling.
Takes about 2–4 minutes without an API key.

In [ ]:
!python -m tools.collect_citations --n-papers 40 --per-paper 2
from google.colab import files
files.download("data/realcite/raw/candidates.jsonl")

## Experiment H: citation audit on real citations (gate + tuned audit threshold)

Needs `data/realcite/raw/candidates.jsonl` from the collection step (upload it if this is a new runtime). Scores three
audits on the held-out half of the real-citation set: no gate, gate, and gate + LLM claim extraction, with the audit's
threshold tuned on the other half. **The labels are unreviewed drafts**: the run prints a warning, and the numbers are
for checking the pipeline, not for the report, until a teammate has reviewed them.

In [ ]:
!python -m eval.run_realcite --name realcite_audit --model {HUB_MODEL} --tune-tau --claim-extractor Qwen/Qwen3-4B
# after the labels are reviewed, add --promote-audit to put the gate and threshold into the app

## Round 2: retrain the verifier on the learned selector's sentences, re-run the real-citation audit

Round 1 showed the selector finds the right sentences far more often (0.94 vs 0.79) but verdicts barely moved,
because the verifier was trained on word-overlap sentences. This trains a verifier on what the selector picks (same
settings as A), and re-runs the real-citation audit with quote grounding that tolerates PDF artifacts and a tau tuner
that keeps the deployed tau unless it clearly beats it. About 25 minutes.

In [ ]:
!python -m eval.finetune_nli --out {MODELS}/ft-base-sel --epochs 3 --batch-size 16 --lr 2e-5 \
    --sentence-scorer cross:{MODELS}/selector && \
  python -m eval.run_eval --name finetuned_nli_selector_v2 --retriever bm25 --verifier nli --model {MODELS}/ft-base-sel \
    --sentence-scorer cross:{MODELS}/selector --oracle --compare-with eval/results/finetuned_nli \
    --compare-with eval/results/finetuned_nli_selector
!python -m eval.run_realcite --name realcite_audit_v2 --model {HUB_MODEL} --tune-tau --claim-extractor Qwen/Qwen3-4B

## Optional ablations (one change at a time, each its own run name)

* `--no-match-inference`: train on gold rationales only (the Session 5 recipe) to measure what the inference-style pairs add
* `--gold-doc-neutral 1`: add non-rationale sentences of gold papers as neutral (may cut false-SUPPORT, may cost recall)
* `--neg-per-claim 2`: more retrieved hard negatives

In [ ]:
# 4. results: the table of every run, and what to bring back to the repo
print(open("eval/results/LATEST.md").read())
import json
for run in ["zero_shot_nli", "finetuned_nli", "zero_shot_nli_large", "finetuned_nli_large",
            "zero_shot_nli_abstract", "finetuned_nli_abstract", "zero_shot_nli_sentence", "finetuned_nli_sentence",
            "hybrid_nli", "hybrid_nli_checks", "hybrid_nli_llm",
            "finetuned_nli_selector", "finetuned_nli_selector_m2"]:
    p = f"eval/results/{run}/results.json"
    if os.path.exists(p):
        r = json.load(open(p)); cl = r["claim_level"]
        print(f"{run:24s} macro-F1 {cl['macro_f1']:.3f} {[round(x, 3) for x in cl['macro_f1_ci95']]}  "
              f"false-SUPPORT {cl['false_support_rate']:.3f}  tau {r['config']['tau']}  "
              + (f"  oracle-paper {r['error_attribution']['oracle_doc']:.3f}" if "error_attribution" in r else "") + "  "
              + "  ".join(f"vs {c['vs']} {c['diff']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}]" for c in r["paired_comparisons"]))

In [ ]:
# 5. download the results; each fine-tuned run's train_log.json is already in eval/results/<run>/
!zip -qr results.zip eval/results
from google.colab import files
files.download("results.zip")


## Before shipping a model

Only promote (`--promote`) if the paired CI vs `zero_shot_nli` is above zero **and** false-SUPPORT stays under the
0.15 ceiling in `docs/decision_rules.json`. The app then needs the checkpoint: upload it to the Hugging Face Hub
and promote the run with `--model <hub-id>` so `deployed_config.json` points at something the app can download.
If the fine-tuned model doesn't win, that is a result too: report it.